In [12]:
# !pip install transformers datasets torch

#!pip install accelerate -U

# !pip install transformers[torch]

# !pip install tf-keras


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


zsh:1: no matches found: transformers[torch]


In [3]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

# Load the pre-trained BERT tokenizer and model for sequence classification
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Set up a sentiment analysis pipeline
nlp = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)


2024-09-13 13:12:24.048458: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/opt/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Sample movie reviews
texts = [
    "I love this movie. It's absolutely fantastic!",
    "This was the worst movie I've ever seen. Total waste of time.",
    "It was okay, not the best but not the worst either.",
    "The acting was bad, but the plot was good."
]

# Perform sentiment analysis
for text in texts:
    result = nlp(text)
    print(f"Review: {text}")
    print(f"Sentiment: {result}\n")


Review: I love this movie. It's absolutely fantastic!
Sentiment: [{'label': 'LABEL_0', 'score': 0.6734955310821533}]

Review: This was the worst movie I've ever seen. Total waste of time.
Sentiment: [{'label': 'LABEL_0', 'score': 0.7042932510375977}]

Review: It was okay, not the best but not the worst either.
Sentiment: [{'label': 'LABEL_0', 'score': 0.6925606727600098}]

Review: The acting was bad, but the plot was good.
Sentiment: [{'label': 'LABEL_0', 'score': 0.6832767128944397}]



In [5]:
result

[{'label': 'LABEL_0', 'score': 0.6832767128944397}]

In [8]:
from transformers import pipeline

# Load the pre-trained BERT model fine-tuned on SST-2 for sentiment analysis
nlp = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Sample movie reviews
texts = [
    "I love this movie. It's absolutely fantastic!",
    "This was the worst movie I've ever seen. Total waste of time.",
    "It was okay, not the best but not the worst either.",
    "The acting was bad, but the plot was good."
]

# Perform sentiment analysis
for text in texts:
    result = nlp(text)
    print(f"Review: {text}")
    print(f"Sentiment: {result[0]['label']} (score: {result[0]['score']:.2f})\n")


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Review: I love this movie. It's absolutely fantastic!
Sentiment: POSITIVE (score: 1.00)

Review: This was the worst movie I've ever seen. Total waste of time.
Sentiment: NEGATIVE (score: 1.00)

Review: It was okay, not the best but not the worst either.
Sentiment: POSITIVE (score: 0.90)

Review: The acting was bad, but the plot was good.
Sentiment: POSITIVE (score: 1.00)



**Training with custom dataset**

In [2]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments


2024-09-13 13:36:23.208005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
?Trainer

Init signature:
Trainer(
    model: Union[transformers.modeling_utils.PreTrainedModel, torch.nn.modules.module.Module] = None,
    args: transformers.training_args.TrainingArguments = None,
    data_collator: Optional[transformers.data.data_collator.DataCollator] = None,
    train_dataset: Union[torch.utils.data.dataset.Dataset, torch.utils.data.dataset.IterableDataset, ForwardRef('datasets.Dataset'), NoneType] = None,
    eval_dataset: Union[torch.utils.data.dataset.Dataset, Dict[str, torch.utils.data.dataset.Dataset], ForwardRef('datasets.Dataset'), NoneType] = None,
    tokenizer: Optional[transformers.tokenization_utils_base.PreTrainedTokenizerBase] = None,
    model_init: Optional[Callable[[], transformers.modeling_utils.PreTrainedModel]] = None,
    compute_metrics: Optional[Callable[[transformers.trainer_utils.EvalPrediction], Dict]] = None,
    callbacks: Optional[List[transformers.trainer_callback.TrainerCallback]] = None,
    optimizers: Tuple[torch.optim.optimizer.Optimizer,

In [4]:

# Load the IMDB dataset
# IMDB (internet movie database) Dataset of 50K Movie Reviews
dataset = load_dataset('imdb')

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the dataset
def tokenize_function(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Remove columns that are not needed for training
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

# Load the pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    evaluation_strategy="epoch",     # evaluate each epoch
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=8,    # batch size for evaluation
    num_train_epochs=1,              # number of epochs
    weight_decay=0.01,               # strength of weight decay
)

# Initialize the Trainer with the model, training arguments, and datasets
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
)

# Fine-tune the model
trainer.train()


/opt/anaconda3/lib/python3.11/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


  0%|          | 0/9375 [00:00<?, ?it/s]

KeyboardInterrupt: 